# 09 — Spectrum Encoder & Differential k-mer Scoring

**Vignette index:** | [`01` GKMKernel basics](01_basic_kernel_matrix.ipynb) | [`02` Distance metrics & kernels](02_distance_metrics_and_kernels.ipynb) | [`03` SVM with kernel](03_svc_with_kernel.ipynb) | [`04` Clustering](04_clustering_sequences.ipynb) | [`05` Long sequence scoring](05_score_long_sequence.ipynb) | [`06` Weighted (WGKM) kernel](06_weighted_kernel.ipynb) | [`07` Transforms & comparison](07_transform_and_comparison.ipynb) | [`08` Windowed 3D tensors](08_windowed_3d_tensors.ipynb) | `**09**` Spectrum encoder & NB | [`10` Gappy encoder](10_gappy_encoder.ipynb) | [`11` Mismatch encoder](11_mismatch_encoder.ipynb) | [`12` Shuffler & chunker](12_shuffler_and_chunker.ipynb)

This vignette demonstrates:
1. Using `SpectrumEncoder` to convert sequences into a sparse CSR matrix of k-mer counts.
2. Using `DifferentialKmerScorer` (Multinomial Naive Bayes) to identify enriched k-mers, with auto-generated negatives via `KmerShuffler`.

**See also:** [12_shuffler_and_chunker.ipynb](12_shuffler_and_chunker.ipynb) for background-model details.

In [1]:
import numpy as np
from kmer.encoders import SpectrumEncoder
from kmer.models import DifferentialKmerScorer
from kmer.perturb import KmerShuffler

## 1. Encode sequences with SpectrumEncoder

In [2]:
seqs = ['ACGTACGTACGT', 'TTTTAAAACCCC', 'GCGCGCGCGCGC']
enc = SpectrumEncoder(k=4, canonical_rc=True)
X = enc.fit_transform(seqs)
print('Shape:', X.shape)
print('Format:', X.format)
print('Feature names[:5]:', enc.get_feature_names_out()[:5])
print('Total counts per seq:', X.sum(axis=1).A1)

Shape: (3, 256)
Format: csr
Feature names[:5]: ['AAAA' 'AAAC' 'AAAG' 'AAAT' 'AACA']
Total counts per seq: [18 18 18]


## 2. Differential k-mer scoring

In [3]:
np.random.seed(0)
positives = ['ACGTACGT' + 'ACGT' * np.random.randint(0, 3) for _ in range(50)]

scorer = DifferentialKmerScorer(
    featurizer=SpectrumEncoder(k=4),
    background=KmerShuffler(k=1, seed=42),
)
scorer.fit(positives)

print('Top 5 enriched k-mers:')
print(scorer.kmer_scores_.sort_values(ascending=False).head(5))

Top 5 enriched k-mers:
ACGT    3.360375
GTAC    2.933857
TACG    2.933857
CGTA    2.751535
ATAA    0.000000
dtype: float64


In [4]:
test_seqs = ['ACGTACGTACGTACGT', 'TTTTAAAAGGGGCCCC', 'GCGCGCGCGCGCGCGC']
scores = scorer.decision_function(test_seqs)
for s, sc in zip(test_seqs, scores):
    print(f'  {s}: {sc:.3f}')

  ACGTACGTACGTACGT: 39.299
  TTTTAAAAGGGGCCCC: -5.375
  GCGCGCGCGCGCGCGC: -11.444


## 3. Effect of background stringency

In [5]:
for bg_k in [1, 2]:
    s = DifferentialKmerScorer(
        featurizer=SpectrumEncoder(k=4),
        background=KmerShuffler(k=bg_k, seed=42),
    )
    s.fit(positives)
    top = s.kmer_scores_.sort_values(ascending=False).head(3)
    print(f'background k={bg_k}: top 3 = {top.to_dict()}')

background k=1: top 3 = {'ACGT': 3.3603753871418998, 'GTAC': 2.9338568698359033, 'TACG': 2.9338568698359033}
background k=2: top 3 = {'AAAA': 0.0, 'AAAC': 0.0, 'AAAG': 0.0}
